# Phase 1 experiments: SGLang 0.5.23 v3
Dedicated SGLang notebook for Qwen3.5, with coupled cache/scheduler sampling,
optional inference parameters, and measured-window server/client telemetry.

Uses versioned `sglang_policy_v3.py`, `runtime_sglang_v3.py`, and
`telemetry_sglang_v3.py`. Previous notebooks and shared Python sources are unchanged.
Run setup cells in order in a fresh kernel. GPU work runs remotely in Jupyter;
CPU simulation checks wiring only and cannot estimate new parameter benefits.
The batch Python entry point does not inherit this v3 configuration.

Weight precision and KV-cache precision remain independent: SGLang uses
`--quantization=fp8` and `--kv-cache-dtype=fp8_e4m3`, with BF16 compute.
Do not approve precision/backend combinations without remote quality checks.


## Differences from the v2 lab
See [the parity checklist](phase1_v2_parity.md). Independent optional cells below restore Pareto/constraint searches, additional pruners, calibration diagnostics, and pinned HF/speculative setup. The independent scalar, Pareto and pruning experiment cells remain available. Untuned baselines and full v2 reporting remain separate. Phase A defaults to the same target as B; explicit HF setup changes only Phase B.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "llm_optimizer").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from llm_optimizer.experiment.phase1 import plot_group
from llm_optimizer.experiment.sglang_policy_v3 import (
    SGLangRunnerV3, DEFAULT_OPTIONAL_PARAMS, native_command,
    runtime_help, audit_command, verify_runtime,
)
from llm_optimizer.experiment.telemetry_sglang_v3 import DEFAULT_TELEMETRY
import copy


## VM, model, workload, and budget controls
This v3 notebook runs SGLang only on its dedicated GPU VM. Use a separate campaign name for independent replications. Changing any search setting creates a new configuration hash; restarts with identical settings resume existing studies. All campaigns on a VM must share `RUN_ROOT` and the same GPU assignment.
`SAMPLER_NAMES` selects GP (`gp`), TPE (`tpe`), and multivariate TPE (`tpe_multivariate`). GP requires the engine-compatible PyTorch installation; unavailable GP is explicitly reported and recorded as skipped, never replaced by TPE. Each combination of sampler, seed, framework, concurrency, objective and pruner has **20 attempted trials** by default. Failed/pruned attempts count toward that budget. Three samplers and three seeds therefore mean 180 attempts for a single concurrency/objective/framework/pruner, not 20 total. Optional `sampler_names=["gp"]` on a group call restricts that cell.

| Concurrency | Requests per trial | Default trial attempts per study |
|---|---|---|
| 1 | 32 | 20 |
| 8 | 62 | 20 |
| 16 | 125 | 20 |
| 32 | 250 | 20 |
| 128 | 1000 | 20 |

The loaded request/concurrency ratio is approximately 1000/128. Concurrency 1 deliberately uses 32 requests. `REQUESTS_BY_CONCURRENCY` is used by both A and B; explicit validation counts remain independently configured at 1000. Pruning observation floors scale from v2's 400/1000 ratio: 12, 24, 50, 100, 400 respectively. With small request counts, tail-latency estimates are noisy; the separate validation budget remains necessary.


In [ ]:
CAMPAIGN = "phase1-sglang-v3"
RUN_NAME = "fp8-clean-retry-01"  # W&B display-name prefix; keep campaign unchanged
SAVE_REQUEST_OUTPUTS = True  # persist generated_texts in local *-measured.jsonl files
FRAMEWORKS = ["sglang"]  # Dedicated SGLang v3 runner
RUN_REAL_GPU = False  # Set True only on the remote GPU VM
CONCURRENCIES = [1, 8, 16, 32, 128]
SEEDS = [11, 29, 47]
RUN_ROOT = ROOT / "experiment_results" / "phase1"
CAMPAIGN_HOURS = 168
GROUP_TIMEOUT_SECONDS = 6 * 3600  # includes all frameworks/seeds in a group
MAX_RESTARTS = 2  # bounded retries after a crashed/timed-out group
MODEL = "Qwen/Qwen3.5-4B"  # same model across A/B isolates search-space effects
MODEL_SNAPSHOT = "/workspace/models/pinned-model-snapshot"
TRIALS = 20  # attempts PER sampler/seed/framework/concurrency/pruner
SAMPLER_NAMES = ["tpe", "tpe_multivariate"]  # select any nonempty subset
PARETO_SAMPLERS = [*SAMPLER_NAMES, "nsga2"]  # available to optional Pareto calls
REQUESTS_BY_CONCURRENCY = {1: 32, 8: 62, 16: 125, 32: 250, 128: 1000}
# Preserve the v2 400/1000 pruning-observation fraction at each request budget.
PRUNING_MIN_COMPLETED_REQUESTS = {
    c: max(1, int(n * 0.4)) for c, n in REQUESTS_BY_CONCURRENCY.items()
}
SAMPLER_SETTINGS = {
    "startup_single": 4, "startup_multi": 10,
    "n_ei_candidates": 24, "population_size": 8,
}
VALIDATION_REPEATS = 5

# W&B controls match optuna_inference_optimization_lab_v2.
WANDB_ENABLED = True
WANDB_PROJECT = "inference-optimization"
WANDB_ENTITY = "maorblumberg-tel-aviv-university"  # or None for your SDK default
WANDB_MODE = "online"  # "offline" keeps runs local; False above disables the SDK
WANDB_GROUP = CAMPAIGN  # same label across framework VMs

# Fill with actual verified values. Never mark checks true without performing them.
PREFLIGHT = {
    framework: {
        "gpu_type": "REPLACE", "framework_version": "REPLACE",
        "cuda_version": "REPLACE", "driver_version": "REPLACE",
        "model_revision": "REPLACE_WITH_MODEL_COMMIT",
        "native_config_verified": False, "quality_verified": False,
        "verified_precision_combinations": [],  # weights/cache pairs approved on this VM
    }
    for framework in ["sglang"]
}


## W&B login and study logging
Install the SDK with `pip install -e '.[notebook,wandb]'`. Run the login cell before creating studies, as in `optuna_inference_optimization_lab_v2.ipynb`. Use `WANDB_ENABLED=False` to disable tracking, or `WANDB_MODE="offline"` to save runs locally without login; upload offline runs later with `wandb sync`.

The existing shared study logger records sampled parameters (including precision), rendered engine configs, workload/environment contracts, benchmark/objective metrics, request accounting, failure/pruning details, cumulative GPU cost, and final best/Pareto candidates. One W&B run is created per study invocation, named after its stable Optuna study name and grouped by `WANDB_GROUP`. Synthetic runs carry the `SYNTHETIC` tag. W&B logging failures warn and allow optimization to continue. Independent validation remains in local artifacts, matching v2.

For unattended runs, authenticate on the VM first with `wandb login` or provide `WANDB_API_KEY` through the service environment. Keep credentials outside notebook cells and experiment configs. The batch supervisor authenticates before launching the queue; CPU smoke tests disable tracking.


In [ ]:
if WANDB_ENABLED and WANDB_MODE == "online":
    import wandb
    wandb.login()  # uses existing credentials/WANDB_API_KEY or prompts interactively


## Explicit base and extended search spaces
Every domain below is editable. Base numeric domains must be regularly spaced and contain at least two values. Extended categorical domains must be supported by the installed engine and model. Remove unsupported choices after preflight, including numerical quality checks for cache dtype changes. Graph profiles and padding are conditional. `schedule_conservativeness` is SGLang-only; Phase A retains its fixed baseline value of 1.0, while Phase B searches the explicit domain below.

In [ ]:
# Illustrative SLOs; replace after measuring the pinned model/workload/GPU.
TTFT_LIMITS_MS = {"1": 100.0, "128": 200.0}
E2E_LIMITS_MS = {"1": 2000.0, "128": 6000.0}
FAILURE_RATE_LIMIT = 0.0
CONSTRAINED = False
PRUNER_NAMES = ["nop", "median", "percentile", "halving", "hyperband", "threshold"]
PRUNING_SAMPLES = 32
# Ordinary serving is explicit. Setup is opt-in and may download checkpoints.
HF_SETUP = {
    "target_model_id": MODEL, "target_revision": "REPLACE_WITH_MODEL_COMMIT",
    "framework": "sglang", "enable_dflash": False,
    "speculative_algorithm": None, "draft_model_id": None, "draft_revision": None,
    "block_size": 16, "num_speculative_tokens": 15, "max_draft_len": 3,
}
# Speculative serving remains a separate campaign with a pinned compatible draft.
# Configure a separate campaign/run root to compare ordinary and speculative serving.
MODEL_PROFILES = {}  # populated only by explicit setup_pinned_models()
MODEL_SETUP_IDENTITY = None
MODEL_SETUP_CHOICES = None
PHASE_B_MODEL = None
PHASE_B_SNAPSHOT = None


In [ ]:
PREFILL_BUDGETS = list(range(1024, 8193, 1024))
TOKEN_BUDGETS = list(range(1024, 32769, 256))
CAPACITIES = list(range(8, 129, 8))
BASE_SPACES = {1: {'sglang': {'chunked_prefill_size': [1024, 2048, 3072, 4096, 5120, 6144, 7168, 8192],
                'cuda_graph_on': [0, 1]}},
 128: {'sglang': {'max_running_requests': [8,
                                           16,
                                           24,
                                           32,
                                           40,
                                           48,
                                           56,
                                           64,
                                           72,
                                           80,
                                           88,
                                           96,
                                           104,
                                           112,
                                           120,
                                           128],
                  'chunked_prefill_size': [1024,
                                           1280,
                                           1536,
                                           1792,
                                           2048,
                                           2304,
                                           2560,
                                           2816,
                                           3072,
                                           3328,
                                           3584,
                                           3840,
                                           4096,
                                           4352,
                                           4608,
                                           4864,
                                           5120,
                                           5376,
                                           5632,
                                           5888,
                                           6144,
                                           6400,
                                           6656,
                                           6912,
                                           7168,
                                           7424,
                                           7680,
                                           7936,
                                           8192,
                                           8448,
                                           8704,
                                           8960,
                                           9216,
                                           9472,
                                           9728,
                                           9984,
                                           10240,
                                           10496,
                                           10752,
                                           11008,
                                           11264,
                                           11520,
                                           11776,
                                           12032,
                                           12288,
                                           12544,
                                           12800,
                                           13056,
                                           13312,
                                           13568,
                                           13824,
                                           14080,
                                           14336,
                                           14592,
                                           14848,
                                           15104,
                                           15360,
                                           15616,
                                           15872,
                                           16128,
                                           16384,
                                           16640,
                                           16896,
                                           17152,
                                           17408,
                                           17664,
                                           17920,
                                           18176,
                                           18432,
                                           18688,
                                           18944,
                                           19200,
                                           19456,
                                           19712,
                                           19968,
                                           20224,
                                           20480,
                                           20736,
                                           20992,
                                           21248,
                                           21504,
                                           21760,
                                           22016,
                                           22272,
                                           22528,
                                           22784,
                                           23040,
                                           23296,
                                           23552,
                                           23808,
                                           24064,
                                           24320,
                                           24576,
                                           24832,
                                           25088,
                                           25344,
                                           25600,
                                           25856,
                                           26112,
                                           26368,
                                           26624,
                                           26880,
                                           27136,
                                           27392,
                                           27648,
                                           27904,
                                           28160,
                                           28416,
                                           28672,
                                           28928,
                                           29184,
                                           29440,
                                           29696,
                                           29952,
                                           30208,
                                           30464,
                                           30720,
                                           30976,
                                           31232,
                                           31488,
                                           31744,
                                           32000,
                                           32256,
                                           32512,
                                           32768],
                  'mem_fraction_static': [0.7, 0.75, 0.8, 0.85]}}}

EXPANDED_CHOICES = {'sglang': {'attention_backend': ['triton', 'fa3', 'flashinfer'],
            'disable_radix_cache': [False, True],
            'schedule_policy': ['fcfs', 'lpm'],
            'enable_mixed_chunk': [False, True],
            'disable_overlap_schedule': [False, True],
            'cuda_graph_backend_decode': ['disabled', 'full'],
            'cuda_graph_backend_prefill': ['disabled', 'breakable']}}

EXTRA_SPACES = {'sglang': {'schedule_conservativeness': [0.5, 0.75, 1.0, 1.25, 1.5],
            'max_prefill_tokens': [8192, 16384, 24576, 32768]}}



## Optional v3 search parameters and measured-window telemetry
All optional dimensions are available but disabled initially to preserve the
20-attempt budget. Enable a small verified subset before constructing studies.
Changing options, telemetry, or versioned implementation files creates a fresh
campaign hash. CPU simulation does not model these new kernels or telemetry.

- GDN decode/prefill kernels: Triton or FlashInfer; independent of standard attention.
- Decode captures: sparse powers of two plus effective capacity, or dense 1..capacity.
  Concurrency 1 fixes capture to [1]; disabled decode graphs skip the dimension.
- Prefill cap: stable categorical choices 4, 16, capacity, clamped to effective
  capacity. `prefill_request_cap_choice` is sampled; `prefill_max_requests` is native.
- KV splits: 4/8/16, sampled only under Triton full-attention decode.
- Prefill interval: 0/2/4, fixed 0 at concurrency 1; minimum batch is derived as
  min(8, concurrency) unless its optional domain is enabled. This can worsen TTFT.

Metrics use the existing measured-request start/stop boundaries, excluding warmup,
startup and rest. Server running/queued counts are sampled gauges, not client
concurrency. Measured client requests in flight count coroutines after semaphore
admission, including transport time. RPS is completed requests per benchmark second
(`request_throughput`), separate from output-token throughput.

Totals exclude priority breakdowns and TP/PP/EP nonzero rank copies. Independent
DP replicas are summed for request counts; pool ratios use the maximum replica
ratio, not a sum. Unknown label layouts become null with a reason. Attention KV
uses `full_token_usage`; `token_usage` is the busiest full/SWA/recurrent pool;
`mamba_usage` is recurrent-state utilization. Pool fractions can include retained
cached state, not just active requests. GPU utilization is NVML device busy percent,
not memory utilization or kernel occupancy. Set an explicit physical NVML index;
CUDA_VISIBLE_DEVICES is not assumed to match physical NVML indices.

Time averages use left-hold integration over covered time, with holds expiring at
2 sampling intervals. Each metric records coverage, sample count, units, reason,
and aggregation; unsupported data is null. Server publication cadence may be
coarser than the scrape interval. No samples are attributed before the first
successful scrape. Peaks are sampled peaks. HTTP/NVML failures do not fail trials.

Telemetry persists in benchmark results JSONL, execution sidecars, Optuna attributes
and W&B. Failed/pruned attempts retain partial execution telemetry even if no
successful benchmark row exists. Collectors stop before server cleanup.


In [ ]:
# Enable only the remotely verified dimensions you want in this campaign.
# Disabled entries do not add dimensions to the 20-attempt search.
optional_params = {'linear_attn_decode_backend': {'enabled': False, 'values': ['triton', 'flashinfer']},
 'linear_attn_prefill_backend': {'enabled': False, 'values': ['triton', 'flashinfer']},
 'decode_capture_profile': {'enabled': False, 'values': ['sparse', 'dense']},
 'prefill_max_requests': {'enabled': False, 'values': [4, 16, 'capacity']},
 'triton_attention_num_kv_splits': {'enabled': False, 'values': [4, 8, 16]},
 'prefill_decode_interval': {'enabled': False, 'values': [0, 2, 4]},
 'prefill_decode_interval_min_bs': {'enabled': False, 'values': ['auto']}}
TELEMETRY = {
    "enabled": True,
    "interval_s": 0.5,
    "timeout_s": 0.25,
    "gpu_index": None,  # Set the verified physical NVML device index for GPU utilization.
}


## Bind the shared runner
Persistent SQLite storage is local to each VM. Copy artifacts off the VM periodically. A VM-wide file lock prevents overlapping workers under the same root. The runner records abandoned RUNNING trials as FAIL after acquiring exclusive ownership; failed and pruned attempts count toward the configured trial budget. Notebook kernels do not have the batch script’s process-group watchdog. After an interrupted notebook, ensure its server processes have stopped before resuming.
Every group invocation reads the current controls and resumes via SQLite. Editing the request map or sampler selection produces a separate campaign configuration hash.


## Weight precision and attention KV-cache precision
Phase B independently samples BF16/FP8 weights and BF16/FP8 attention KV cache.
Phase A uses BF16 weights and default BF16 attention KV. Compute/nonquantized
layers remain BF16. Online `quantization="fp8"` loads the unquantized pinned
checkpoint; it does not export a new checkpoint.

FP8 KV uses `fp8_e4m3`. KV scales default to 1.0 unless provided by checkpoint
metadata or a validated `quantization_param_path` in fixed server configuration.
This setting does not quantize Qwen3.5's separate GDN recurrent state.
Check output quality, recurrent-state dtype, and every selected backend on the GPU.
Record approved pairs in `PREFLIGHT["sglang"]["verified_precision_combinations"]`.
Random fixed-length performance prompts are not a quality evaluation.

[0.5.23 quantization](https://github.com/sgl-project/sglang/blob/v0.5.23/docs/docs/advanced_features/quantization.mdx)
[0.5.23 KV cache](https://github.com/sgl-project/sglang/blob/v0.5.23/docs/docs/advanced_features/quantized_kv_cache.mdx)


In [ ]:
PRECISION_CHOICES = {
    "sglang": {"weight_precision": ["bf16", "fp8"],
               "kv_cache_precision": ["bf16", "fp8"]},
}


## JSON export and response capture
Run the next cell before constructing runners. It fixes JSON exports containing `inf`,
`-inf` or `nan` without modifying the stored Optuna trials. To finish the previous studies,
set `SAVE_REQUEST_OUTPUTS=False` and use exactly the previous campaign/model configuration.

With `SAVE_REQUEST_OUTPUTS=True` (the default here), generated text is retained after each
saved benchmark in `*-measured.jsonl`, under `results.generated_texts`. This setting changes
the frozen configuration and creates a fresh results directory under the same campaign.
Changing `RUN_NAME` alone does not reset studies. Text from previous runs cannot be recovered.
Responses are saved locally, not automatically uploaded to W&B, and are available after the
benchmark returns rather than streamed live. Attempts interrupted by errors or pruning may
not produce a result row. The random-token workload is for performance, not quality validation.


In [ ]:
# Notebook-only JSON export fix: preserves SQLite values; sanitizes exported copies.
import json
import math
from pathlib import Path
import llm_optimizer.experiment.phase1 as phase1_module
import llm_optimizer.experiment.workflow as workflow_module

def clean_export_json(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {key: clean_export_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_export_json(item) for item in value]
    return value

def safe_write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(clean_export_json(value), indent=2, allow_nan=False) + "\n"
    )
    temporary.replace(path)

# phase1 imported write_json by name, so patch both references in this kernel.
phase1_module.write_json = safe_write_json
workflow_module.write_json = safe_write_json


In [ ]:
# v3 uses the correct shared SGLang precision mapping directly.
SGLANG_V3_COMPATIBILITY = {"adapter": "sglang-v3", "framework_version": "0.5.23"}


In [ ]:
def make_runner(for_model_setup=False):
    if MODEL_SETUP_IDENTITY is not None and MODEL_SETUP_IDENTITY != HF_SETUP:
        raise ValueError("HF_SETUP changed; clear MODEL_PROFILES and MODEL_SETUP_IDENTITY, then rerun setup")
    if MODEL_SETUP_CHOICES is not None and MODEL_SETUP_CHOICES != EXPANDED_CHOICES[HF_SETUP["framework"]]:
        raise ValueError("Expanded choices changed after HF setup; clear setup profiles and rerun setup")
    settings = {
        "ttft_limits_ms": TTFT_LIMITS_MS, "e2e_limits_ms": E2E_LIMITS_MS,
        "failure_rate_limit": FAILURE_RATE_LIMIT, "pruning_samples": PRUNING_SAMPLES,
        "constrained": CONSTRAINED,
        "wandb_enabled": WANDB_ENABLED,
        "wandb_project": WANDB_PROJECT, "wandb_entity": WANDB_ENTITY,
        "wandb_mode": WANDB_MODE, "wandb_group": WANDB_GROUP,
        "frameworks": FRAMEWORKS, "real_frameworks": FRAMEWORKS,
        "run_real_gpu": RUN_REAL_GPU and not for_model_setup, "quick_demo": False,
        "phases": ["A", "B"], "concurrencies": CONCURRENCIES,
        "seeds": SEEDS, "sampler_names": SAMPLER_NAMES, "pareto_samplers": PARETO_SAMPLERS,
        "phase_models": {"A": MODEL, "B": PHASE_B_MODEL or MODEL},
        "model_snapshots": {"A": MODEL_SNAPSHOT, "B": PHASE_B_SNAPSHOT or MODEL_SNAPSHOT},
        # Lab's phase-level fallback; actual study counts use the map below.
        "phase_requests": {"A": REQUESTS_BY_CONCURRENCY[128],
                           "B": REQUESTS_BY_CONCURRENCY[128]},
        "sampler_settings": SAMPLER_SETTINGS,
        "search_trials": {p: {"1": TRIALS, "128": TRIALS} for p in ["A", "B"]},
        "base_workload": {
            **({"output_details": True} if SAVE_REQUEST_OUTPUTS else {}),
            "dataset_name": "random", "apply_chat_template": True, "random_input_len": 512,
            "random_output_len": 128, "random_range_ratio": 1.0,
            "seed": 2026, "request_rate": "inf",
        },
        "validation_repeats": VALIDATION_REPEATS,
        "validation_requests": {"A": 1000, "B": 1000},
        "gpu_count": 1, "rest_seconds": 10,
        "real_preflight": {
            phase: {f: {**PREFLIGHT[f], "expanded_choices": EXPANDED_CHOICES[f],
                           **(MODEL_PROFILES.get(f, {}) if phase == "B" else {})}
                    for f in FRAMEWORKS}
            for phase in ["A", "B"]
        },
    }
    return SGLangRunnerV3(
        settings, BASE_SPACES, EXPANDED_CHOICES, EXTRA_SPACES, RUN_ROOT,
        precision_choices=PRECISION_CHOICES,
        optional_params=optional_params, telemetry_options=TELEMETRY,
        requests_by_concurrency=REQUESTS_BY_CONCURRENCY,
        pruning_min_completed_requests=PRUNING_MIN_COMPLETED_REQUESTS,
    )

# Construct from the current controls for each invocation; SQLite resumes studies.
def get_runner():
    return make_runner()

def run_objective_group(phase, target, concurrencies, pruner_name="nop", sampler_names=None, constrained=None):
    global LAST_RUN_DIR
    runner = get_runner()
    LAST_RUN_DIR = runner.run_dir
    print("Campaign:", CAMPAIGN, "| W&B run prefix:", RUN_NAME)
    print("Results directory:", LAST_RUN_DIR)
    print("Concurrency / full-trial requests:",
          [(c, REQUESTS_BY_CONCURRENCY[c]) for c in concurrencies])
    return runner.run_objective_group(
        phase, target, concurrencies, pruner_name, sampler_names=sampler_names,
        constrained=CONSTRAINED if constrained is None else constrained,
    )

def setup_pinned_models():
    """Explicit GPU setup; commit resolved profiles into the editable controls."""
    global PHASE_B_MODEL, PHASE_B_SNAPSHOT, MODEL_SETUP_IDENTITY, MODEL_SETUP_CHOICES
    if not RUN_REAL_GPU:
        raise ValueError("Set RUN_REAL_GPU=True before model setup")
    runner = make_runner(for_model_setup=True)
    runner.lab.RUN_REAL_GPU = True
    with runner.lock():
        summary = runner.lab.configure_hf_models(**HF_SETUP)
    framework = HF_SETUP["framework"]
    import copy
    profile = runner.lab.REAL_PREFLIGHT["B", framework]
    # Verification declarations remain editable in PREFLIGHT after setup.
    MODEL_PROFILES[framework] = {
        key: copy.deepcopy(profile[key]) for key in (
            "model_revision", "hf_target_model_id", "fixed_server", "draft_model_id",
            "draft_revision", "speculative_decoding", "framework_version", "cuda_version",
            "driver_version", "gpu_type",
        ) if key in profile
    }
    if profile["expanded_choices"] != EXPANDED_CHOICES[framework]:
        MODEL_PROFILES[framework]["expanded_choices"] = copy.deepcopy(profile["expanded_choices"])
    MODEL_SETUP_CHOICES = copy.deepcopy(EXPANDED_CHOICES[framework])
    PHASE_B_MODEL, PHASE_B_SNAPSHOT = summary["target_model_id"], summary["target_path"]
    MODEL_SETUP_IDENTITY = dict(HF_SETUP)
    return summary


In [ ]:
# Preserve W&B naming while keeping the versioned logger's telemetry additions.
_original_make_runner = getattr(make_runner, "_without_sglang_v3_names", make_runner)
def make_runner(for_model_setup=False):
    PREFLIGHT["sglang"]["notebook_compatibility"] = dict(SGLANG_V3_COMPATIBILITY)
    runner = _original_make_runner(for_model_setup=for_model_setup)
    original_logger = runner.lab.NotebookWandbLogger
    run_prefix = RUN_NAME
    def named_logger(study, contract):
        logger = original_logger(study, contract)
        if logger.tracker is not None and run_prefix:
            logger.tracker.run.name = f"{run_prefix}-{study.study_name}"
        return logger
    runner.lab.NotebookWandbLogger = named_logger
    return runner
make_runner._without_sglang_v3_names = _original_make_runner


In [ ]:
# CPU wiring assertions only; no server launch or quality approval.
from types import SimpleNamespace
from llm_optimizer.experiment.phase1 import ExplicitLab
for weights in ("bf16", "fp8"):
    for cache in ("bf16", "fp8"):
        config = ExplicitLab.precision_config(SimpleNamespace(), "sglang", weights, cache)
        assert config.get("quantization") == ("fp8" if weights == "fp8" else None)
        assert config["kv_cache_dtype"] == ("fp8_e4m3" if cache == "fp8" else "auto")
print("SGLang precision wiring verified; remote loading/quality checks remain required.")


## Resolve a pinned Hugging Face model for SGLang
Run these cells after the configuration and runner cells. Setup downloads or reuses the model snapshot and requires a CUDA GPU. Choose an immutable commit for reproducible reruns; `main` resolves to its current commit. The setup below selects ordinary serving.


In [ ]:
import tempfile
from pathlib import Path

hf_cache = Path(tempfile.mkdtemp(prefix="hf-cache-", dir="/tmp"))
HF_SETUP["cache_dir"] = str(hf_cache)

print("Hugging Face cache:", hf_cache)
print("Writable:", hf_cache.is_dir())

In [ ]:
verify_runtime()  # Verify the exact release before GPU model setup/downloads.
FRAMEWORKS = ["sglang"]
RUN_REAL_GPU = True
MODEL_PROFILES = {}
MODEL_SETUP_IDENTITY = None
MODEL_SETUP_CHOICES = None

HF_SETUP.update(
    framework="sglang", target_model_id=MODEL, target_revision="main",
    enable_dflash=False, speculative_algorithm=None,
    draft_model_id=None, draft_revision=None,
)
hf_models = setup_pinned_models()
hf_models


In [ ]:
MODEL = hf_models["target_model_id"]
MODEL_SNAPSHOT = hf_models["target_path"]
for key in ("gpu_type", "framework_version", "cuda_version",
            "driver_version", "model_revision"):
    PREFLIGHT["sglang"][key] = MODEL_PROFILES["sglang"][key]
print("Pinned snapshot:", MODEL_SNAPSHOT)
print("Pinned revision:", PREFLIGHT["sglang"]["model_revision"])


In [ ]:
MODEL = hf_models["target_model_id"]
MODEL_SNAPSHOT = hf_models["target_path"]

# Record the resolved runtime and revision for Phase A.
for key in (
    "gpu_type",
    "framework_version",
    "cuda_version",
    "driver_version",
    "model_revision",
):
    PREFLIGHT["sglang"][key] = MODEL_PROFILES["sglang"][key]

print("Phase A snapshot:", MODEL_SNAPSHOT)
print("Phase B snapshot:", PHASE_B_SNAPSHOT)

## SGLang 0.5.23 runtime and command audit
Run on the remote VM after setup. Check the exact package version and use the
kernel interpreter for server launch. Audit native flags including enabled
optional parameters; CLI presence does not prove GPU/backend compatibility.
The cache-policy search contains only cache-on/FCFS, cache-on/LPM, and cache-off/FCFS.


In [ ]:
import optuna
print(verify_runtime())
SGLANG_HELP = runtime_help()
audit_runner = make_runner(for_model_setup=True)
audit_lab = audit_runner.lab
for concurrency in CONCURRENCIES:
    menu = audit_lab.sampling_menu("sglang", concurrency, "expanded")
    print("Concurrency:", concurrency, "sampling menu:", menu)
    # Audit every categorical value independently, including inactive branches.
    baseline = {key: values[0] for key, values in menu.items()}
    if "triton" in menu["attention_backend"]:
        baseline["attention_backend"] = "triton"
    if "full" in menu["cuda_graph_backend_decode"]:
        baseline["cuda_graph_backend_decode"] = "full"
    if "prefill_decode_interval" in menu:
        baseline["prefill_decode_interval"] = max(menu["prefill_decode_interval"])
    for name, values in menu.items():
        for value in values:
            config = audit_lab.suggest_config(optuna.trial.FixedTrial({**baseline, name: value}),
                                              "sglang", concurrency, "expanded")
            config = {**MODEL_PROFILES["sglang"].get("fixed_server", {}), **config}
            if TELEMETRY["enabled"]:
                config["enable_metrics"] = True
            command = native_command(audit_lab, config, MODEL_SNAPSHOT, concurrency)
            audit_command(command, SGLANG_HELP)
print("Selected command flag names passed audit; GPU smoke and quality checks are separate.")


In [ ]:
# Optional remote smoke: one real server, no Optuna study.
RUN_SGLANG_SMOKE = False
SMOKE_WEIGHTS, SMOKE_CACHE = "bf16", "bf16"
SMOKE_PORT = 30000
if RUN_SGLANG_SMOKE:
    import requests
    from llm_optimizer.server_utils import start_server, terminate_process_top_down
    smoke_runner = make_runner(for_model_setup=True)
    smoke_lab = smoke_runner.lab
    smoke_config = smoke_lab.render_engine_config("sglang", 8, {
        "max_running_requests": 8, "chunked_prefill_size": 2048, "mem_fraction_static": 0.75})
    smoke_config.update(smoke_lab.precision_config("sglang", SMOKE_WEIGHTS, SMOKE_CACHE))
    smoke_config.update(cuda_graph_backend_decode="disabled",
                        cuda_graph_backend_prefill="disabled", enable_metrics=True)
    smoke_config = {**MODEL_PROFILES["sglang"].get("fixed_server", {}), **smoke_config}
    command = native_command(smoke_lab, smoke_config, MODEL_SNAPSHOT, 8, SMOKE_PORT)
    audit_command(command, SGLANG_HELP)
    print(command)
    with smoke_runner.lock():
        process = None
        try:
            process = start_server(command, {}, f"http://127.0.0.1:{SMOKE_PORT}/health", False)
            response = requests.post(f"http://127.0.0.1:{SMOKE_PORT}/v1/chat/completions",
                json={"model": str(MODEL_SNAPSHOT), "messages": [
                    {"role": "user", "content": "What is 17 times 19? Explain briefly."}],
                    "temperature": 0, "max_tokens": 128}, timeout=180)
            response.raise_for_status()
            print(response.json()["choices"][0]["message"])
            from llm_optimizer.experiment.telemetry_sglang_v3 import aggregate_prometheus
            metrics = requests.get(f"http://127.0.0.1:{SMOKE_PORT}/metrics", timeout=10)
            metrics.raise_for_status()
            print("Available gauges:", aggregate_prometheus(metrics.text))
        finally:
            if process is not None:
                terminate_process_top_down(process)
    print("Review text and metrics. This does not approve optional kernel/graph combinations.")


Update SGLang verification declarations only after remote native-command,
model-loading and numerical quality checks. Repeat smoke checks for every selected
precision/backend combination, including enabled optional kernels. FlashInfer GDN
requires compatible hardware/state dtype; SM100+ prefill requires CUDA 13+.


In [ ]:
print("SGLang version:", MODEL_PROFILES["sglang"]["framework_version"])
print("Expanded choices:", EXPANDED_CHOICES["sglang"])
print("Enabled optional parameters:", {k: v for k, v in optional_params.items() if v["enabled"]})
# Narrow precision choices if only a subset has passed quality checks.
PREFLIGHT["sglang"].update({
    "native_config_verified": False,
    "quality_verified": False,
    "verified_precision_combinations": [
        # "bf16/bf16", "bf16/fp8", "fp8/bf16", "fp8/fp8",
    ],
})
print("Preflight:", PREFLIGHT["sglang"])


## Independent experiment cells
Day 1: preflight, untuned baseline measurements, and recovery drills. Days 2–4: base (A) versus extended (B) searches on the same model, then median versus no pruning. Days 5–6: independent repeats and workload checks. Day 7: sustained-load confirmation. The queue below performs the search stage; use the validation cell afterwards. Measure trial duration during the pilot and adjust trial counts to fit the week. A finite completed queue exits successfully rather than repeating identical experiments indefinitely.

### A: single e2e (no pruning)

In [ ]:
A_single_e2e_studies = run_objective_group(
    "A", "e2e", [1], pruner_name="nop",
)
plot_group(A_single_e2e_studies)


### A: single ttft (no pruning)

In [ ]:
A_single_ttft_studies = run_objective_group(
    "A", "ttft", [1], pruner_name="nop",
)
plot_group(A_single_ttft_studies)


### A: loaded throughput (no pruning)

In [ ]:
A_loaded_throughput_studies = run_objective_group(
    "A", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_throughput_studies)


### A: loaded e2e (no pruning)

In [ ]:
A_loaded_e2e_studies = run_objective_group(
    "A", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_e2e_studies)


### B: single e2e (no pruning)

In [ ]:
B_single_e2e_studies = run_objective_group(
    "B", "e2e", [1], pruner_name="nop",
)
plot_group(B_single_e2e_studies)


### B: single ttft (no pruning)

In [ ]:
B_single_ttft_studies = run_objective_group(
    "B", "ttft", [1], pruner_name="nop",
)
plot_group(B_single_ttft_studies)


### B: loaded throughput (no pruning)

In [ ]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_throughput_studies)


### B: loaded e2e (no pruning)

In [ ]:
B_loaded_e2e_studies = run_objective_group(
    "B", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_e2e_studies)


### B: loaded throughput with median pruning
The first ten trials establish pruning history. Decisions use measured request checkpoints, with at least three reference trials. The no-pruning comparison above uses the same workload and seeds.

In [ ]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1],
    pruner_name="median",
)
plot_group(B_loaded_throughput_studies)


## View and export saved model responses
Run this cell after any experiment has saved a benchmark result. By default it shows every
response from the most recently modified result file's last benchmark. Select a file and row
to inspect an earlier trial. It also writes a readable text copy alongside the JSONL file.


In [ ]:
# Run after a benchmark finishes; rerun to inspect newer saved results.
# Optional: paste a prior campaign directory or an exact *-measured.jsonl path.
RESULTS_DIR = None
RESULT_FILE = None
RESULT_INDEX = -1  # last saved benchmark in the selected file
WRITE_READABLE_COPY = True

if RESULT_FILE is not None:
    output_path = Path(RESULT_FILE)
else:
    output_dir = Path(RESULTS_DIR) if RESULTS_DIR is not None else globals().get("LAST_RUN_DIR")
    if output_dir is None:
        raise ValueError("Run an experiment first, or set RESULTS_DIR / RESULT_FILE above")
    candidates = list(Path(output_dir).glob("*-measured.jsonl"))
    output_path = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

if output_path is None:
    print("No saved benchmark results yet.")
else:
    rows = []
    for line in output_path.read_text().splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print("Skipping an incomplete/invalid JSONL row; rerun after the writer finishes.")
    if not rows:
        print("No complete result rows yet.")
    else:
        row = rows[RESULT_INDEX]
        details = row.get("results", {})
        texts = details.get("generated_texts", [])
        print("File:", output_path, "| selected row:", RESULT_INDEX, "| rows:", len(rows))
        print("Trial:", row.get("optimization", {}).get("trial_number"))
        if not texts:
            print("This result contains no saved text; enable SAVE_REQUEST_OUTPUTS for future runs.")
        else:
            errors = details.get("errors", [])
            blocks = []
            for i, text in enumerate(texts):
                block = f"--- Request {i + 1} ---\n{text}"
                if i < len(errors) and errors[i]:
                    block += f"\nError: {errors[i]}"
                blocks.append(block)
            readable = "\n\n".join(blocks)
            print(readable)
            if WRITE_READABLE_COPY:
                row_number = RESULT_INDEX if RESULT_INDEX >= 0 else len(rows) + RESULT_INDEX
                readable_path = output_path.with_name(
                    f"{output_path.stem}-row{row_number}-generations.txt")
                readable_path.write_text(readable + "\n")
                print("Readable copy saved:", readable_path)


## Independent candidate validation
Run after search, within the remaining GPU budget. This uses the repository’s shortlist and independent validation workload/repeats; it is not an automatic sustained-load soak test. Keep the serving stack pinned and record any separate soak workload. Validation in this interactive cell does not use the batch watchdog.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    from llm_optimizer.experiment.workflow import load_studies, write_json
    runner = get_runner()
    with runner.lock():
        studies = load_studies(runner.lab)
        validation = runner.lab.validate_candidates(runner.lab.shortlist(studies))
        write_json(runner.run_dir / "validation.json", runner.lab.artifact_value(validation))


## Execution boundary
Use this notebook for v3 behavior. The existing batch Python entry point retains
its previous behavior and does not inherit `optional_params` or v3 telemetry.
Notebook kernels do not have the batch process-group watchdog. After an interrupted
kernel, verify that its server processes stopped before resuming.


## Optional Pareto, constraint, and pruning experiments
Run these cells independently after editing SLOs, samplers and pinned serving settings. Active pruning is supported only for unconstrained scalar objectives. Calibration uses the same task workload and scaled request checkpoints; pilot samples are additional to the 20-attempt comparison budgets. Diagnostics replay a first-checkpoint median rule and estimate savings; validate them on the GPU before trusting pruning. Threshold searches reuse a persisted task calibration.


In [ ]:
pareto_latency = run_objective_group("B", "pareto_latency", [1])
plot_group(pareto_latency)


In [ ]:
pareto_loaded = run_objective_group("B", "pareto_throughput", [c for c in CONCURRENCIES if c > 1])
plot_group(pareto_loaded)


In [ ]:
constrained_scalar = run_objective_group("B", "e2e", [1], constrained=True)
plot_group(constrained_scalar)


In [ ]:
constrained_pareto = run_objective_group("B", "pareto_latency", [1], constrained=True)
plot_group(constrained_pareto)


In [ ]:
diagnostics = get_runner().calibrate("B", "throughput", 8, FRAMEWORKS[0], samples=PRUNING_SAMPLES)
diagnostics


In [ ]:
pruning_comparison = {name: run_objective_group("B", "throughput", [8], pruner_name=name)
                      for name in PRUNER_NAMES}
